# Adaptive Scaffolding of Embodyment-Aware Concept Learning using Meta Learning and Knowledge Graphs

## Ideas for the implementation

- [ ] Teacher policy is a directed graph.
    - Nodes of the graph represent skills; direction of the edges and distance between the nodes represents the conditioning and similarity of the skills, resp.
- [ ] In each step simulated student learns by choosing a graph node, given current knowledge state.
    - The node generates a reward sampled from a distribution, which is  parameterized by both the distance from the current knowledge state, and the embodiment of the student.
    - We accumulate the reward in each node as long as the node is marked as "learned".
    - Open question:
        - [ ] how should the student pick the nodes for the next learning?
        - [ ] should that also be controlled by a distribution?
        - [ ] other better ideas?
- [ ] The scaffolding agent learns from the student interactions and has a capacity to model an optimized learning policy, given students' individual learning trajectories.
- [ ] Coordinator manages between the student and the scaffolding agent based on the reward (or are there better options?)
- [ ] In this implementation there is no explicit teacher guidance when the student gets stuck. The guidance is implicitly implemented with the teacher graph.

### Additional ToDos

- [ ] Define a consistent terminolofy for all building blocks
  - E.g. How to call the Learning-Agent of the coordinator? Again *scaffolding agent* as in [Moringen et.al 2024](https://openreview.net/forum?id=Tx6nIfIkLH)?


In [112]:
!pip install gym


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [113]:
!pip install sklearn

^C

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
ERROR: Operation cancelled by user


In [223]:
import random
import numpy as np
import networkx as nx
import gymnasium as gym
from gymnasium import spaces
from sklearn.preprocessing import LabelEncoder

## Define the Semantic Knowledge Graph

Create a class that is able to represent a semantic knowledge graph. The graph should be directed and should allow adding concepts and relationships between them. The graph should also allow querying properties such as neighbors, learned status, and distances.

> Idea 💡: The knowledge graph should be part of the student and also integrated in the environment!


In [224]:
class SemanticKnowledgeGraph:
    """A class representing a semantic knowledge graph.

    This graph is implemented as a directed graph using NetworkX. It allows
    adding concepts, defining relationships between them, and querying
    properties such as neighbors, learned status, and distances.

    Attributes:
        graph (networkx.DiGraph): The directed graph representing the knowledge
            graph.
    """
    def __init__(self) -> None:
        """Initializes the semantic knowledge graph."""
        self.graph = nx.DiGraph()

    def add_concept(self, concept: str) -> None:
        """Adds a concept to the graph.

        Args:
            concept (str): The concept to add.
        """
        self.graph.add_node(concept, learned=False, accumulated_reward=0)

    def add_relationship(self, concept1: str, concept2: str) -> None:
        """Adds a directed relationship between two concepts.

        Args:
            concept1 (str): The first concept.
            concept2 (str): The second concept.
        """
        self.graph.add_edge(concept1, concept2)

    def get_neighbors(self, concept: str) -> list[str]:
        """Gets the neighbors of a concept.

        Args:
            concept (str): The concept to query.

        Returns:
            list: A list of neighboring concepts.
        """
        return list(self.graph.successors(concept))

    def mark_learned(self, concept: str, marker: bool = True) -> None:
        """Marks a concept as learned.

        Args:
            concept (str): The concept to mark as learned.
            marker (bool): The marker to set. Defaults to True.
        """
        self.graph.nodes[concept]['learned'] = marker

    def is_learned(self, concept: str) -> bool:
        """Checks if a concept is learned.

        Args:
            concept (str): The concept to check.

        Returns:
            bool: True if the concept is learned, False otherwise.
        """
        return self.graph.nodes[concept].get('learned', False)

    def get_distance(self, start: str, end: str) -> float:
        """Calculates the shortest path distance between two concepts.

        If no directed path exists, it attempts to calculate the distance
        using an undirected version of the graph.

        Args:
            start (str): The starting concept.
            end (str): The target concept.

        Returns:
            float: The shortest path distance, or infinity if no path
            exists.
        """
        try:
            return nx.shortest_path_length(self.graph, source=start, target=end)
        except nx.NetworkXNoPath:
            # Use undirected shortest path if no directed path exists
            undirected_graph = self.graph.to_undirected()
            try:
                return nx.shortest_path_length(undirected_graph, source=start, target=end)
            except nx.NetworkXNoPath:
                return float('inf')

## Define the Student

The student represents the simulated learner in the environment. Its goal is to learn all concepts that are defined via the knowledge graph.
It has a knowledge state that keeps track of the concepts learned. The student can query the graph to determine the reward for learning a new concept based on its distance from the current concept and its talent distribution.

In [225]:
# Define the Student
class Student:
    """Represents a student learning concepts in a graph-based environment.

    Attributes:
        knowledge_state (set): The set of concepts the student has learned.
        talent_distribution (dict): Maps concepts to their difficulty levels.
        graph (nx.DiGraph): The graph representing the relationships between concepts.
        rewards (list): A list of rewards accumulated during learning.

    Methods:
        query(current_concept, concept):
            Calculates the reward for attempting to learn a concept based on
            distance and talent distribution.

        learn(concept):
            Adds a concept to the knowledge state if sufficient reward is
            accumulated.
    """
    def __init__(self, talent_distribution: dict, graph: nx.DiGraph) -> None:
        self.knowledge_state = set()
        self.talent_distribution = talent_distribution  # Dict mapping concept to difficulty
        self.graph = graph
        self.rewards = []

    def reset(self) -> None:
        """Resets the student's knowledge state."""
        self.knowledge_state = set()
        self.rewards = []
        for node in self.graph.graph.nodes:
            self.graph.graph.nodes[node]['accumulated_reward'] = 0

    def query(self, current_concept: str, concept: str) -> float:
        """Calculates the reward for attempting to learn a concept.

        Args:
            current_concept (str): The current concept the student is focused on.
            concept (str): The target concept to learn.

        Returns:
            float: The reward for attempting to learn the concept.
        """
        if concept in self.knowledge_state:
            return 0  # Already learned
        distance = self.graph.get_distance(current_concept, concept)
        distance_factor = max(0.1, 1 / (distance + 1))  # Closer nodes have higher rewards
        reward = np.random.beta(2, self.talent_distribution.get(concept, 2)) * distance_factor  # Talent + distance
        self.graph.graph.nodes[concept]['accumulated_reward'] += reward
        self.rewards.append(reward)
        return reward

    def learn(self, concept: str) -> None:
        """Adds a concept to the knowledge state if sufficient reward is accumulated.

        Args:
            concept (str): The concept to be learned.
        """
        if concept not in self.knowledge_state:
            if self.graph.graph.nodes[concept]['accumulated_reward'] >= 1:  # Accumulate enough reward to learn
                self.knowledge_state.add(concept)
                self.graph.mark_learned(concept)

## Learning Environment

The learning environment is defined using the OpenAI Gym framework. It simulates the interaction between the student and the knowledge graph. The environment allows the student to take actions (learn concepts) and receive rewards based on their learning progress.

> **Note:** The environment is designed to be compatible with reinforcement learning algorithms, allowing for the integration of various RL agents.

The environment is initialized with a graph and a student. The `reset` method initializes the environment, while the `step` method executes an action and returns the new state, reward, and whether the learning process is complete.

### TODO

The learning environment consists of a Knowledge Graph and one or more students that try to learn the concepts that are defined by the graph.
Thus, we should rewrite the enviroment, that it fully handles the creation and control of the student(s), as well as the knowledge graph.
In this way, we can generate a general environment interface.


In [259]:
# Define the RL Environment
class LearningEnv(gym.Env):
    """Gym environment for learning concepts in a knowledge graph.

    This environment models a learning process where a student interacts
    with a graph of concepts. The student learns by transitioning between
    concepts and receiving rewards based on their learning progress.

    Attributes:
        graph (nx.DiGraph): A graph representing the relationships between concepts.
        student (Student): An agent that learns concepts from the graph.
        __current_concept (str): The concept the student is currently learning.
        action_dim (int): The number of possible actions (concepts).
        state_dim (int): The number of possible states (concepts).
        action_space (spaces.Discrete): The set of possible actions (concepts).
            It is equal to the observation space.
    """
    def __init__(self, graph: nx.DiGraph, talent_distribution: dict) -> None:
        """Initializes the LearningEnv with a graph and a student.

        Args:
            graph (Graph): The graph of concepts.
            talent_distribution (dict): Maps concepts to their difficulty levels.
        """

        super(LearningEnv, self).__init__()

        self.graph = graph

        self.student = Student(talent_distribution, graph)
        self.action_dim = len(self.graph.graph.nodes)
        self.state_dim = len(self.graph.graph.nodes)


        # 5eed an encode that maps nodes to integers for Learner
        self.state_encoder = LabelEncoder()
        print(self.graph.graph.nodes)
        self.state_encoder.fit(list(self.graph.graph.nodes))

        self.__current_concept = self.random_concept

        self.action_space = spaces.Discrete(len(self.graph.graph.nodes))

    @property
    def random_concept(self) -> int:
        """Return random concept.

        Return one random concept from the knowledge graph.
        1. Select one random node
        2. Transform it to integer representation

        Returns:
            int: Integer representation of selected concept.
        """
        return self.state_encoder.transform([np.random.choice(list(self.graph.graph.nodes))])[0]

    @property
    def neighbors(self) -> list[int]:
        """Gets the neighbors of the current concept.

        This method retrieves the neighboring concepts of the current concept
        in the graph and encodes them as integers using the state encoder.

        Returns:
            list[int]: A list of encoded integers representing the neighbors
            of the current concept.
        """
        return self.state_encoder.transform(self.graph.get_neighbors(self.state_encoder.inverse_transform([self.__current_concept])[0]))

    def reset(self, seed: int = None, options: dict =None) -> tuple[int, dict]:
        """Resets the environment to its initial state.

        Also reset the student and the knowledge graph.

        Args:
            seed (int, optional): A seed for random number generation.
            options (dict, optional): Additional options for resetting.

        Returns:
            tuple: Initial state and an empty dictionary.
        """
        super().reset(seed=seed)
        self.student.reset()
        for c in list(self.graph.graph.nodes):
            self.graph.mark_learned(c, False)

        self.__current_concept = self.random_concept
        return self.__current_concept, {}

    def decode_state(self, state: int) -> str:
        """Decodes the state index back to the concept name.

        Args:
            state (int): The state index.

        Returns:
            str: The corresponding concept name.
        """
        return self.state_encoder.inverse_transform([state])[0]

    def step(self, action: int) -> tuple[int, int, float, bool, bool, dict]:
        """Executes a step in the environment based on the given action.

        Args:
            action (int): The index of the next concept to transition to.

        Returns:
            tuple: A tuple containing:
                - action (int): The action taken as it is the new state.
                - reward (float): The reward received for the action.
                - done (bool): Whether the learning process is complete.
                - False (bool): Placeholder for compatibility.
                - dict: An empty dictionary for additional info.
        """
        print("executing step with ", action)
        decoded_current_concept = self.state_encoder.inverse_transform([self.__current_concept])[0]
        decoded_next_concept = self.state_encoder.inverse_transform([action])[0]
        print("current concept", decoded_current_concept)
        print("next_concept", decoded_next_concept)
        reward = self.student.query(decoded_current_concept, decoded_next_concept)
        self.student.learn(decoded_next_concept)
        self.__current_concept = next_concept
        done = all(self.graph.is_learned(c) for c in self.graph.graph.nodes)
        return action, reward, done, False, {}

## Learning Agent

Agent that learns to coordinate the learning process. It is used in the `Coordinator` class to manage the student's learning behavior.
The agent uses a Dyna-Q algorithm, which combines Q-learning with planning.

In [227]:
class DynaQAgent:
    """DynaQAgent.

    A Dyna-Q agent for reinforcement learning that combines Q-learning
    with planning.

    Attributes:
    ----------
    n_states (int): Number of states in the environment.
    n_actions (int): Number of possible actions in the environment.
    epsilon (float): Probability of choosing a random action (exploration).
    alpha (float): Learning rate for updating Q-values.
    gamma (float): Discount factor for future rewards.
    planning_steps (int): Number of planning steps to perform.
    q_table (ndarray): Q-values table for state-action pairs.
    model (dict): Model to store state transitions and rewards.
    """

    def __init__(
        self,
        n_states: int,
        n_actions: int,
        epsilon: float = 0.1,
        alpha: float = 0.1,
        gamma: float = 0.95,
        planning_steps: int = 5,
    ) -> None:
        """Initialize the DynaQAgent with the given parameters.

        Args:
        n_states (int): Number of states in the environment.
        n_actions (int): Number of possible actions in the environment.
        epsilon (float): Probability of choosing a random action (exploration).
        alpha (float): Learning rate for updating Q-values.
        gamma (float): Discount factor for future rewards.
        planning_steps (int): Number of planning steps to perform.

        Returns:
        None
        """
        self.n_states = n_states
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps

        # Initialize Q-table and model
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}

    def predict(self, state: int) -> int:
        """Choose an action.

        Choose an action based on the current state using an epsilon-greedy
        policy.

        Args:
        state (int): The current state of the environment.

        Returns:
        int: The action chosen, either randomly (exploration) or based on the
        highest Q-value (exploitation).
        """
        if random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.n_actions)
        return np.argmax(self.q_table[state])

    def update(self, state: int, action: int, reward: float, next_state: int) -> None:
        """Update Model.

        Update the Q-table and model with the given transition and perform
        planning steps.

        Args:
        state (int): The current state of the environment.
        action (int): The action taken from the current state.
        reward (float): The reward received after taking the action.
        next_state (int): The state transitioned to after taking the action.

        Returns:
        None
        """
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error

        # Store the transition in the model
        self.model[(state, action)] = (reward, next_state)

        # Planning phase
        for _ in range(self.planning_steps):
            s, a = random.choice(list(self.model.keys()))
            r, s_next = self.model[(s, a)]
            best_next_a = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_a]
            td_error = td_target - self.q_table[s][a]
            self.q_table[s][a] += self.alpha * td_error

## Coordinator

The `Coordinator` class coordinates the learning strategy of the student.
It is a wrapper around the Q-Learning agent,
manages and updates its Q-values based on the student's learning progress.

Thus, it can query the next concept to lern by utilizing different strategies.

These strategies are:

- **Random Neighbor**: The next concept is chosen randomly from the neighbors of the current concept.
- **Reinforcement Learning (RL)**: The next concept is chosen based on the Q-values learned by the agent.


The `Coordinator` class can switch between these stratgies after *every learning attempt (i.e. step)*. The currently implemented rule when to use the *reinforcement learning strategy* triggers when the average reward of the last 10 learning attempts is below a threshold (0.05). This means that the student is not making sufficient progress in learning the concepts.

### TODO

- Implement a more sophisticated strategy for switching between the two strategies.
- Maybe implement more strategies
- Align the rule when to switch strategies to [Moringen et.al 2024](https://openreview.net/forum?id=Tx6nIfIkLH)



In [228]:
class Coordinator:
    """Coordinates the interaction between the student and the environment.

    This class manages the strategy for teaching the student, either using
    reinforcement learning (RL) or a predefined strategy. It also handles
    the agent's training and decision-making process.

    Attributes:
        student: The student object being trained.
        graph: The graph structure representing the knowledge domain.
        use_rl_strategy: A boolean indicating whether to use RL-based strategy.
        agent: The RL agent used for decision-making.
    """
    def __init__(self, state_dim: int, action_dim: int) -> None:
        """Initializes the Coordinator with a student and a graph.

        Args:
            student: The student object being trained.
            graph: The graph structure representing the knowledge domain.
        """
        self.use_rl_strategy = False
        # Initialize the agent with the environment's state and action space
        self.agent = DynaQAgent(state_dim,action_dim)

    def switch_strategy(self, student: Student) -> None:
        """Switches the teaching strategy based on the student's progress.

        If the student's recent progress is below a threshold, switches to
        an RL-based strategy.

        Args:
            student: The student object being trained.
        """
        if (len(student.rewards)>10):
            print (np.mean(student.rewards[-10:]))
            progress = np.mean(student.rewards[-10:])

            self.use_rl_strategy = progress < 0.05  # If progress is low, switch to RL-based strategy

    def give_data_to_agent(self, state, action, reward, next_state, done):
        """Provides experience data to the RL agent and trains it.

        Args:
            state: The current state of the environment.
            action: The action taken by the agent.
            reward: The reward received after taking the action.
            next_state: The state of the environment after the action.
            done: A boolean indicating if the episode has ended.
        """
        print("Giving data to agent:", state, action, reward, next_state)
        self.agent.update(state, action, reward, next_state)

    def query_next_concept(self, env: gym.Env, concept_state: int) -> int:
        """Determines the next concept to teach based on the current strategy.

        Args:
            env: The environment object representing the teaching context.
            concept_state: The current state of the student.

        Returns:
            The next concept to teach.
        """
        self.switch_strategy(env.student)
        if self.use_rl_strategy:
            #if self.agent is None:
                #self.train_agent(env)  # Train the agent if it hasn't been trained yet
            print(f"Agent predicting action in state {concept_state}")
            action = self.agent.predict(concept_state)
        else:
            #Student has own strategy
            neighbors = env.neighbors
            action = np.random.choice(neighbors) if len(neighbors) else np.random.choice(range(env.state_dim))
        return action


# Experiments


## Setup Learning Environment

The environment consists of the following building Blocks

- **Semantic Knowledge Graph**
  - Graph that defines the to-be-learned-concepts and their connections
- **Talent Distribution (Emobodyment)**
  - The distribution characterizes the student's individual talents and determines how successfull it is in learning specific concepts.
- **Student**
  - The entitity that has the *goal to learn all conecpts in the knowledge graph*.
- **Coordinator**
  - The core-unit of the whole experiment
  - Coordinates the learning process of the student by interventing and helping (using the Learning-Agent) when it deems it necesseray
  - Implements Learning-Agent that learns, based on the students learning-behavior, which concepts to learn next.

In [229]:
# Setup the Environment
graph = SemanticKnowledgeGraph()
concepts = ['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']
for c in concepts:
    graph.add_concept(c)
graph.add_relationship('Math', 'Algebra')
graph.add_relationship('Algebra', 'Calculus')
graph.add_relationship('Physics', 'Mechanics')
print (graph.graph.nodes)

['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']


In [230]:
# This should be different for differently embodied students
talent_distribution = {'Math': 2, 'Algebra': 3, 'Calculus': 5, 'Physics': 4, 'Mechanics': 6}

In [231]:
env = LearningEnv(graph, talent_distribution)

['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']


In [232]:
coordinator = Coordinator(env.state_dim, env.action_dim)

## Trainig

> 🎯The *student learns all concepts in the knowledge graph* using a *minimal number of attempts*

It runs but there is no actual lerning

**TODO:**
- [ ] Plots of average reward of student
- [ ] Plots of average reward of agent
- [ ] Plots of number of steps for learning with
  - random concepts
  - only concepts selected by agent
  - switching strategy based on average reward


In [258]:
# Train agent while student learns
state, _ = env.reset()
print("Start State:", env.decode_state(state))
for step in range(500):
    # Next concept to learn
    next_concept = coordinator.query_next_concept(env, state)
    print("Next concept to learn:", next_concept)
    # Should execute the action in the environment
    # how to get the index of the above action in the list of concepts

    # HACK: This should be the action space of the environment
    action = next_concept
    next_state, reward, done, info, _ = env.step(action)  # Take the action in the environment
    coordinator.give_data_to_agent(state, action, reward, next_state, done)
    state = next_state
    if done:
        print(f"Learning completed after {step} steps!")
        break


Start State: Math
Next concept to learn: 0
executing step with  0
current concept Math
next_concept Algebra
Giving data to agent: 2 0 0.1387894346056651 0
Next concept to learn: 1
executing step with  1
current concept Algebra
next_concept Calculus
Giving data to agent: 0 1 0.21373160212914463 1
Next concept to learn: 0
executing step with  0
current concept Calculus
next_concept Algebra
Giving data to agent: 1 0 0.4278294429648186 0
Next concept to learn: 1
executing step with  1
current concept Algebra
next_concept Calculus
Giving data to agent: 0 1 0.23405952184356246 1
Next concept to learn: 1
executing step with  1
current concept Calculus
next_concept Calculus
Giving data to agent: 1 1 0.060637813689095044 1
Next concept to learn: 1
executing step with  1
current concept Calculus
next_concept Calculus
Giving data to agent: 1 1 0.2143120547664874 1
Next concept to learn: 0
executing step with  0
current concept Calculus
next_concept Algebra
Giving data to agent: 1 0 0.299150351921